In [1]:
# orders # /public/trendytech/retail_db/orders
# ========
# order_id,order_date,order_customer_id,order_status
# 1,2013-07-25 00:00:00.0,11599,CLOSED
# 2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT
# 3,2013-07-25 00:00:00.0,12111,COMPLETE

# customers # /public/trendytech/retail_db/customers
# ==========
# customer_id,customer_fname,customer_lname,customer_email,customer_password,customer_street,customer_city,customer_state,customer_zipcode
# 1,Richard,Hernandez,XXXXXXXXX,XXXXXXXXX,6303 Heather Plaza,Brownsville,TX,78521
# 2,Mary,Barrett,XXXXXXXXX,XXXXXXXXX,9526 Noble Embers Ridge,Littleton,CO,80126
# 3,Ann,Smith,XXXXXXXXX,XXXXXXXXX,3422 Blue Pioneer Bend,Caguas,PR,00725

# order_items # /public/trendytech/retail_db/order_items
# ==========
# order_item_id,order_id,order_item_product_id,order_item_quantity,order_item_subtotal,order_item_product_price
# 1,1,957,1,299.98,299.98
# 2,2,1073,1,199.99,199.99
# 3,2,502,5,250.0,50.0
# 4,2,403,1,129.99,129.99

In [2]:
from pyspark.sql import SparkSession
import getpass
username = getpass.getuser()
spark = SparkSession. \
builder. \
appName("week4-assignments"). \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [3]:
orders = spark.sparkContext.textFile("/public/trendytech/retail_db/orders/part-00000")

In [4]:
#customers = spark.sparkContext.textFile("/public/trendytech/retail_db/customers/part-00000")

In [5]:
order_items = spark.sparkContext.textFile("/public/trendytech/retail_db/order_items/part-00000")

In [6]:
## 1. we need to find top 10 customers who have spent the most amount(premium customers)

In [7]:
orders1 = orders.map(lambda x: (x.split(",")[0],x.split(",")[2]))  #(order_id,order_customer_id)

In [8]:
#orders1.take(3)

In [9]:
order_items1 = order_items.map(lambda x: (x.split(",")[1],float(x.split(",")[4]))) #(order_id,order_item_subtotal)

In [10]:
#order_items1.take(3)

In [11]:
order_cust_item = orders1.join(order_items1)  #### (order_id ,(order_customer_id,order_item_subtotal))

In [12]:
#order_cust_item.take(3)

In [13]:
cust_total = order_cust_item.map(lambda x: x[1]) ### (order_customer_id,order_item_subtotal)

In [14]:
#cust_total.take(3)

In [15]:
cust_total_sum = cust_total.reduceByKey(lambda x,y : x+y).sortBy(lambda x:x[1],False)

In [16]:
cust_total_sum.take(10)

[('791', 10524.170000000002),
 ('9371', 9299.03),
 ('8766', 9296.14),
 ('1657', 9223.71),
 ('2641', 9130.92),
 ('1288', 9019.11),
 ('3710', 9019.099999999999),
 ('4249', 8918.85),
 ('5654', 8904.95),
 ('5624', 8761.98)]